# Pertemuan 2: Gradient Descent

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dafx7/COMPUTING-AI-ML-OPTIMIZATION/blob/main/week2/notebooks/01_gradient_descent.ipynb)

Kita melatih garis `ŷ = w·x + a` pada data baterai Pekan 1 dengan **gradient descent**. Sel pertama berisi fungsi bantu untuk data, grafik, dan pemeriksaan: **jalankan saja**. Semua bagian gradient descent kita tulis sendiri.

Ganti setiap `...` dengan kodemu, lalu jalankan sel pemeriksa di bawahnya.

In [ ]:
# ===== Fungsi bantu: jalankan saja, tidak perlu dibaca =====
import numpy as np
import matplotlib.pyplot as plt
np.seterr(all="ignore")  # nilai yang meledak menjadi inf/nan tanpa pesan peringatan

# Data baterai dari Pekan 1: x dalam PULUHAN menit, y baterai (%)
x = np.array([0, 1, 2, 3, 4], dtype=float)
y = np.array([97, 89, 80, 74, 66], dtype=float)
x_menit = x * 10  # data yang sama, x dalam menit (dipakai di bagian C)


def cek(syarat, pesan="Belum tepat."):
    try:
        benar = bool(syarat())
    except Exception as e:
        benar, pesan = False, f"{pesan} (Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e})"
    print("Benar." if benar else "Belum tepat. " + pesan)


def dekat(a, b, tol=1e-6):
    return abs(a - b) <= tol * max(1.0, abs(b))


def _mse(w, a, xs, ys):
    return float(np.mean((ys - (w * xs + a)) ** 2))


def plot_data(w=None, a=None, xs=None, ys=None):
    xs = x if xs is None else xs; ys = y if ys is None else ys
    plt.figure(figsize=(6, 4))
    plt.scatter(xs, ys, s=60, color="black", zorder=3, label="data")
    if w is not None:
        t = np.linspace(xs.min() - 0.5, xs.max() + 0.5, 50)
        plt.plot(t, w * t + a, color="tab:orange", lw=2.5, label=f"garis: w = {w:.3f}, a = {a:.3f}")
    plt.xlabel("x"); plt.ylabel("baterai (%)"); plt.legend(); plt.grid(alpha=0.3); plt.show()


def plot_loss(riwayat):
    """riwayat: daftar (w, a, mse), atau dict {label: riwayat} untuk membandingkan beberapa."""
    kumpulan = riwayat if isinstance(riwayat, dict) else {"MSE": riwayat}
    plt.figure(figsize=(7, 4))
    for label, r in kumpulan.items():
        m = np.array([t[2] for t in r], dtype=float)
        m[~np.isfinite(m) | (m > 1e12)] = np.nan
        plt.plot(m, lw=2, label=str(label))
    plt.yscale("log"); plt.xlabel("langkah"); plt.ylabel("MSE (skala log)")
    plt.legend(); plt.grid(alpha=0.3); plt.show()


def plot_path(riwayat, xs=None, ys=None):
    xs = x if xs is None else xs; ys = y if ys is None else ys
    P = np.array([(t[0], t[1]) for t in riwayat], dtype=float)
    P = P[np.all(np.isfinite(P), axis=1) & np.all(np.abs(P) < 1e4, axis=1)]
    w_ls, a_ls = np.polyfit(xs, ys, 1)
    ws = np.concatenate([P[:, 0], [w_ls]]); as_ = np.concatenate([P[:, 1], [a_ls]])
    pw, pa = (ws.max() - ws.min()) * 0.15 + 1, (as_.max() - as_.min()) * 0.15 + 1
    W, A = np.meshgrid(np.linspace(ws.min() - pw, ws.max() + pw, 200), np.linspace(as_.min() - pa, as_.max() + pa, 200))
    Z = np.mean((ys[:, None, None] - (W * xs[:, None, None] + A)) ** 2, axis=0)
    plt.figure(figsize=(6, 5))
    plt.contour(W, A, np.log(Z + 1e-9), levels=20, colors="lightgray")
    plt.plot(P[:, 0], P[:, 1], "-o", color="tab:orange", ms=3, lw=1.5, label="langkah GD")
    plt.scatter([w_ls], [a_ls], marker="*", s=250, color="tab:red", zorder=4, label="minimum")
    plt.xlabel("w"); plt.ylabel("a"); plt.legend(); plt.show()


def cek_gradien(fungsi, xs=None, ys=None):
    """Bandingkan gradienmu dengan perkiraan numerik: geser sedikit ke kiri dan ke kanan."""
    xs = x if xs is None else xs; ys = y if ys is None else ys
    rng = np.random.default_rng(0); h = 1e-5
    try:
        for _ in range(5):
            w, a = rng.uniform(-20, 20), rng.uniform(-50, 150)
            gw, ga = fungsi(w, a, xs, ys)
            gw_n = (_mse(w + h, a, xs, ys) - _mse(w - h, a, xs, ys)) / (2 * h)
            ga_n = (_mse(w, a + h, xs, ys) - _mse(w, a - h, xs, ys)) / (2 * h)
            if not (dekat(gw, gw_n, 1e-4) and dekat(ga, ga_n, 1e-4)):
                print(f"Belum tepat. Di w = {w:.2f}, a = {a:.2f}: gradienmu ({gw:.3f}, {ga:.3f}), perkiraan numerik ({gw_n:.3f}, {ga_n:.3f}).")
                return
        print("Benar. Gradienmu cocok dengan perkiraan numerik di 5 titik acak.")
    except Exception as e:
        print(f"Belum tepat. Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e}")


def cek_hasil(w, a, xs=None, ys=None):
    xs = x if xs is None else xs; ys = y if ys is None else ys
    w_ls, a_ls = np.polyfit(xs, ys, 1)
    print(f"GD     : w = {w:.4f}, a = {a:.4f}")
    print(f"Rumus  : w = {w_ls:.4f}, a = {a_ls:.4f}   (least squares, Pekan 1)")
    print("Benar. Sama dengan rumus." if dekat(w, w_ls, 1e-3) and dekat(a, a_ls, 1e-3) else "Belum sama. Tambah langkah atau periksa loop-mu.")


def plot_sweep(hasil):
    """hasil: dict {eta: riwayat}. Menggambar MSE akhir terhadap eta dan mengembalikan eta terbaik."""
    etas = sorted(hasil)
    akhir = []
    print(f"{'eta':>10}  {'MSE akhir':>14}")
    for e in etas:
        m = hasil[e][-1][2]
        ok = np.isfinite(m) and m < 1e12 and m <= hasil[e][0][2]
        akhir.append(m if ok else np.nan)
        print(f"{e:>10g}  {('%.4f' % m) if ok else 'meledak':>14}")
    akhir = np.array(akhir, dtype=float)
    plt.figure(figsize=(7, 4))
    plt.plot(etas, akhir, "-o", color="tab:orange", lw=2)
    for e, m in zip(etas, akhir):
        if np.isnan(m):
            plt.axvline(e, color="tab:red", ls="--", alpha=0.6)
    plt.xscale("log"); plt.yscale("log"); plt.xlabel("eta (skala log)"); plt.ylabel("MSE akhir (skala log)")
    plt.title("garis merah putus-putus: meledak"); plt.grid(alpha=0.3); plt.show()
    if np.all(np.isnan(akhir)):
        print("Semua eta meledak."); return None
    terbaik = etas[int(np.nanargmin(akhir))]
    print(f"eta terbaik di daftar ini: {terbaik}")
    return terbaik

print("Fungsi bantu siap.")

## Bagian A: error dan gradiennya

### A1. MSE

$$\mathrm{MSE}(w, a) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - (w\,x_i + a)\right)^2$$

Dengan NumPy, `y - (w * x + a)` menghitung error semua data sekaligus. Tulis rumusnya persis: `np.sum` adalah $\sum$, dan `/ n` adalah $\frac{1}{n}$, dengan `n = len(x)`.

In [ ]:
def mse(w, a, x=x, y=y):
    n = len(x)
    e = ...
    return ...

In [ ]:
cek(lambda: dekat(mse(0, 0), 6712.4) and dekat(mse(-7.7, 96.6), 0.38), 'mse(0, 0) seharusnya 6712,4 dan mse(-7.7, 96.6) seharusnya 0,38.')

### A2. Gradien

Turunan yang baru kita turunkan di papan, dengan $e_i = y_i - (w\,x_i + a)$:

$$\frac{\partial\,\mathrm{MSE}}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n} x_i\,e_i \qquad \frac{\partial\,\mathrm{MSE}}{\partial a} = -\frac{2}{n}\sum_{i=1}^{n} e_i$$

`cek_gradien` membandingkan hasilmu dengan perkiraan numerik: ia menggeser `w` (dan `a`) sedikit ke kiri dan ke kanan, lalu melihat perubahan MSE. Begitulah cara memeriksa turunan tanpa harus percaya begitu saja.

In [ ]:
def gradien(w, a, x=x, y=y):
    n = len(x)
    e = ...
    gw = ...
    ga = ...
    return gw, ga

In [ ]:
cek_gradien(gradien)

## Bagian B: loop gradient descent

Setiap langkah: hitung kedua gradien dengan `w` dan `a` yang **lama**, lalu perbarui keduanya.

$$w \leftarrow w - \eta\,\frac{\partial\,\mathrm{MSE}}{\partial w} \qquad a \leftarrow a - \eta\,\frac{\partial\,\mathrm{MSE}}{\partial a}$$

Simpan `(w, a, mse)` setelah setiap langkah di `riwayat`, supaya bisa kita gambar.

In [ ]:
def gradient_descent(w, a, eta, langkah, x=x, y=y):
    riwayat = [(w, a, mse(w, a, x, y))]
    for t in range(langkah):
        ...
        ...
        riwayat.append((w, a, mse(w, a, x, y)))
    return riwayat

In [ ]:
cek(lambda: len(gradient_descent(0, 0, 0.1, 5)) == 6 and dekat(gradient_descent(0, 0, 0.1, 1)[1][0], 29.4) and dekat(gradient_descent(0, 0, 0.1, 1)[1][1], 16.24), 'Satu langkah dari (0, 0) dengan eta = 0,1 seharusnya memberi w = 29,4 dan a = 16,24 (hitungan tangan di slide).')

### B2. Jalankan dan lihat hasilnya

In [ ]:
riwayat = gradient_descent(0, 0, eta=0.1, langkah=300)
w_akhir, a_akhir, _ = riwayat[-1]
plot_loss(riwayat)
plot_path(riwayat)
plot_data(w_akhir, a_akhir)
cek_hasil(w_akhir, a_akhir)

### B3. Coba learning rate lain

Bandingkan $\eta$ = 0,01; 0,14; dan 0,15. Apa yang terjadi pada masing-masing?

In [ ]:
plot_loss({
    "eta = 0.01": gradient_descent(0, 0, 0.01, 300),
    "eta = 0.14": gradient_descent(0, 0, 0.14, 300),
    "eta = 0.15": gradient_descent(0, 0, 0.15, 300),
})

## Bagian C: mencari learning rate terbaik

**Arti terbaik:** MSE terkecil setelah anggaran **100 langkah**, mulai dari `w = 0, a = 0`.

### C1. Sapu kasar dengan skala log
Setiap nilai kira-kira 3 kali nilai sebelumnya. Isi `hasil[eta]` dengan riwayat gradient descent untuk setiap `eta`.

In [ ]:
etas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
hasil = {}
for eta in etas:
    hasil[eta] = ...
plot_sweep(hasil)

In [ ]:
cek(lambda: sorted(hasil) == etas and all(len(r) == 101 for r in hasil.values()), 'Setiap eta harus dijalankan 100 langkah dari (0, 0).')

### C2. Perbesar di dekat tebing
Di mana MSE berubah dari sangat kecil menjadi meledak? Pilih 5 sampai 8 nilai `eta` di daerah itu, sapu lagi, dan simpan hasil terbaiknya di `eta_terbaik`.

In [ ]:
etas_zoom = [...]
hasil_zoom = {}
for eta in etas_zoom:
    hasil_zoom[eta] = gradient_descent(0, 0, eta, 100)
eta_terbaik = plot_sweep(hasil_zoom)

In [ ]:
cek(lambda: eta_terbaik is not None and 0.12 <= eta_terbaik < 0.149, 'Untuk data ini eta terbaik ada sedikit di bawah tebing, antara 0,12 dan 0,149. Perbesar lebih rapat di sana.')

### C3. Data yang sama, x dalam menit
Jalankan sapuan yang sama dengan `x_menit` (0, 10, 20, 30, 40). Bandingkan $\eta$ terbaiknya, dan MSE terbaik setelah 100 langkah.

In [ ]:
etas_menit = [0.00001, 0.00003, 0.0001, 0.0003, 0.001, 0.0015, 0.003]
hasil_menit = {}
for eta in etas_menit:
    hasil_menit[eta] = gradient_descent(0, 0, eta, 100, x=x_menit)
plot_sweep(hasil_menit)
w_ls, a_ls = np.polyfit(x_menit, y, 1)
print("Sebagai pembanding, MSE minimum yang mungkin:", round(mse(w_ls, a_ls, x_menit, y), 2))

## Pertanyaan pemahaman (dikumpulkan)

Jawab masing-masing dalam dua sampai tiga kalimat.

1. Mengapa eta terbaik ada tepat di bawah tebing (nilai yang mulai meledak)?
2. Mengapa eta terbaik berubah ketika x ditulis dalam menit, dan apa yang sebaiknya kita lakukan kalau semua eta terasa lambat?

*Jawabanmu:* ...